In [ ]:
# Kaggle runner: control-variate weak-error benchmark (g1-g4, regimes A-E).
#
# CONVENTION NOTE.  Every other notebook in notebooks/kaggle/ is standalone:
# it inlines its kernels so it runs with Internet OFF and no pushed branch
# (see notebooks/kaggle/README.md).  This one deliberately does the opposite
# and clones the repo, so the experiment code has a single home and cannot
# drift from experiments/run_weak_error.py.  The cost is that you must:
#
#     Settings -> Internet -> On          (requires a phone-verified account)
#
# Accelerator is not needed: this experiment is NumPy/CPU.  Set it to None.
#
# RESUMING ACROSS SESSIONS.  /kaggle/working is wiped between sessions, so to
# continue an interrupted run: "Add Input" -> Notebook Output -> pick this
# notebook's previous version.  The cell copies any weak_error_partial.csv it
# finds under /kaggle/input into the working directory and passes --resume.
# Levels are independently seeded, so a resumed run reproduces an
# uninterrupted one bit for bit.
#
# CONFIGURATION (environment variables, all optional):
#   WEAK_CV_RUN_MODE      smoke | full            (default full)
#   WEAK_CV_REPO          clone URL
#   WEAK_CV_REF           branch, tag or commit   (default main)
#   WEAK_CV_REGIMES       e.g. "A B C D E"
#   WEAK_CV_N_PATHS       Monte Carlo paths
#   WEAK_CV_N_STEPS       e.g. "8 16 ... 4096"
#   WEAK_CV_MAX_ADAPTIVE  cap on nominal steps for KLM / adaptive KL
#   WEAK_CV_SCHEMES       subset of FTE HH ProjEuler KL IF KLM BLT
#   WEAK_CV_TIME_BUDGET_S stop cleanly after this many seconds
#   WEAK_CV_NO_CV         "1" to reproduce the old direct estimator

import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "cir_repo"
WORK.mkdir(parents=True, exist_ok=True)

RUN_MODE = os.environ.get("WEAK_CV_RUN_MODE", "full").strip().lower()

PRESETS = {
    # Integrity check: a few minutes, one boundary regime.
    "smoke": dict(
        regimes="E",
        n_paths="5000",
        n_steps="8 16 32 64 128",
        max_adaptive="128",
    ),
    # Production: the deep ladder h = 2^-3 .. 2^-12 for the fixed-step
    # schemes.  The adaptive schemes stop at 2^-10: their accepted-step
    # counts run ~30x the nominal level in regime E, and they are NOT put on
    # a shared fine grid (quantising an adaptive step to a 2^-k grid forces
    # every step to one grid step once h_max reaches the spacing, silently
    # collapsing the scheme to a uniform mesh).
    "full": dict(
        regimes="A B C D E",
        n_paths="200000",
        n_steps="8 16 32 64 128 256 512 1024 2048 4096",
        max_adaptive="1024",
    ),
}
if RUN_MODE not in PRESETS:
    raise SystemExit(f"WEAK_CV_RUN_MODE must be one of {sorted(PRESETS)}")
cfg = dict(PRESETS[RUN_MODE])

cfg["regimes"] = os.environ.get("WEAK_CV_REGIMES", cfg["regimes"])
cfg["n_paths"] = os.environ.get("WEAK_CV_N_PATHS", cfg["n_paths"])
cfg["n_steps"] = os.environ.get("WEAK_CV_N_STEPS", cfg["n_steps"])
cfg["max_adaptive"] = os.environ.get("WEAK_CV_MAX_ADAPTIVE", cfg["max_adaptive"])

REPO_URL = os.environ.get(
    "WEAK_CV_REPO", "https://github.com/lukemurray01/CIR_MSc_2025-26.git"
)
REF = os.environ.get("WEAK_CV_REF", "main")
SCHEMES = os.environ.get("WEAK_CV_SCHEMES", "").strip()

# Kaggle CPU sessions stop at 12 h; leave an hour to write outputs.
TIME_BUDGET = os.environ.get("WEAK_CV_TIME_BUDGET_S", "39600").strip()

# ---------------------------------------------------------------- resume ---
# Seed the working directory from a previous run attached as an input.
resumed_from = None
for candidate in sorted(Path("/kaggle/input").glob("**/weak_error_partial.csv")):
    shutil.copy(candidate, WORK / "weak_error_partial.csv")
    resumed_from = str(candidate)
    break

resume = (WORK / "weak_error_partial.csv").exists()
print(f"run mode      : {RUN_MODE}")
print(f"resume        : {resume}" + (f" (from {resumed_from})" if resumed_from else ""))

# ----------------------------------------------------------------- clone ---
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(
    ["git", "clone", "--quiet", REPO_URL, str(REPO)], check=True
)
subprocess.run(["git", "checkout", "--quiet", REF], cwd=REPO, check=True)
commit = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPO, check=True,
    capture_output=True, text=True,
).stdout.strip()
print(f"repo          : {REPO_URL}")
print(f"ref / commit  : {REF} / {commit}")

# ------------------------------------------------------------------- run ---
cmd = [
    sys.executable, "experiments/run_weak_error.py",
    "--regimes", *cfg["regimes"].split(),
    "--n-paths", cfg["n_paths"],
    "--n-steps", *cfg["n_steps"].split(),
    "--max-adaptive-steps", cfg["max_adaptive"],
    "--out-dir", str(WORK),
]
if SCHEMES:
    cmd += ["--schemes", *SCHEMES.split()]
if TIME_BUDGET:
    cmd += ["--time-budget-s", TIME_BUDGET]
if resume:
    cmd += ["--resume"]
if os.environ.get("WEAK_CV_NO_CV", "").strip() == "1":
    cmd += ["--no-control-variate"]

print("command       : " + " ".join(cmd) + "\n", flush=True)

env = dict(os.environ, PYTHONUNBUFFERED="1", MPLBACKEND="Agg")
completed = subprocess.run(cmd, cwd=REPO, env=env)

# --------------------------------------------------------------- archive ---
# Provenance for the reproducibility ledger: exactly what produced the CSVs.
(WORK / "weak_error_run_config.json").write_text(
    json.dumps(
        {
            "run_mode": RUN_MODE,
            "repo": REPO_URL,
            "ref": REF,
            "commit": commit,
            "command": cmd,
            "resumed": resume,
            "resumed_from": resumed_from,
            "returncode": completed.returncode,
        },
        indent=2,
    ),
    encoding="utf-8",
)

artefacts = [
    p for name in (
        "weak_error.csv", "weak_error_orders.csv", "weak_error_partial.csv",
        "weak_error_run_config.json",
    )
    for p in [WORK / name] if p.exists()
]
artefacts += sorted(WORK.glob("weak_error_regime_*.pdf"))
artefacts += sorted(WORK.glob("weak_error_regime_*.png"))

with zipfile.ZipFile(WORK / "weak_error_results.zip", "w",
                     zipfile.ZIP_DEFLATED) as archive:
    for path in artefacts:
        archive.write(path, path.name)

print("\nartefacts in /kaggle/working:")
for path in artefacts:
    print(f"  {path.name:<42} {path.stat().st_size / 1024:9.1f} KiB")
print("  weak_error_results.zip  <- download this")

if completed.returncode != 0:
    raise SystemExit(f"run_weak_error.py exited {completed.returncode}")
